In [34]:
%pip install skorch torch

In [35]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
from skorch import NeuralNetClassifier

In [36]:
def make_mlp(in_dim, layer_sizes):
    layers = []
    for h in layer_sizes:
        layers += [nn.Linear(in_dim, h), nn.ReLU()]
        in_dim = h
    return nn.Sequential(*layers), in_dim


In [37]:
class ParallelMultiPathNet(nn.Module):
    def __init__(self, input_dim, num_classes,
                 pre_layers=(128, 64),
                 paths=((64, 32), (64, 32, 16, 32)),
                 fusion_dim=128,
                 post_layers=(64, 32),
                 fusion_mode="concat"):
        super().__init__()
        if not 1 < len(pre_layers) <= 5:
            raise ValueError("Need 1 < m <= 5 layers before the parallel module")
        if not 2 <= len(paths) <= 3:
            raise ValueError("Need 2 <= k <= 3 parallel paths")
        if not 1 <= len(post_layers) <= 5:
            raise ValueError("Need 1 <= n + l <= 5 layers after fusion")
        if fusion_mode not in ("concat", "add"):
            raise ValueError("fusion_mode must be 'concat' or 'add'")
        if fusion_mode == "add" and len({p[-1] for p in paths}) != 1:
            raise ValueError("For 'add' fusion, all paths must end with the same width")

        self.fusion_mode = fusion_mode
        self.pre, dim = make_mlp(input_dim, pre_layers)

        self.paths = nn.ModuleList()
        out_dims = []
        for sizes in paths:
            path, out_dim = make_mlp(dim, sizes)
            self.paths.append(path)
            out_dims.append(out_dim)

        fused_in = sum(out_dims) if fusion_mode == "concat" else out_dims[0]
        self.fusion = nn.Sequential(nn.Linear(fused_in, fusion_dim), nn.ReLU())
        self.post, dim = make_mlp(fusion_dim, post_layers)
        self.classifier = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.pre(x)
        outs = [path(x) for path in self.paths]
        x = torch.cat(outs, dim=1) if self.fusion_mode == "concat" else torch.stack(outs).sum(dim=0)
        return self.classifier(self.post(self.fusion(x)))

In [39]:
# Load dataset
digits = load_digits()
X = digits.data
y = digits.target
print("Dataset shape:", X.shape)
print("Number of classes:", len(np.unique(y)))
print("Number of input features:", X.shape[1]) # every image has 64 input features

# Initialize training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("=" * 50)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("=" * 50)

# Scale Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Dataset shape: (1797, 64)
Number of classes: 10
Number of input features: 64
Training samples: 1437
Testing samples: 360


In [38]:
# Initialize Bottleneck Model (sklearn)
Bottleneck = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 128),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

# Initialize Parallel Model (skorch)
Parallel = NeuralNetClassifier(
    ParallelMultiPathNet,
    module__input_dim=X_train.shape[1],
    module__num_classes=len(np.unique(y)),
    module__pre_layers=(128, 64),
    criterion=nn.CrossEntropyLoss,
    optimizer=torch.optim.Adam,
    lr=1e-3,
    max_epochs=50,
    train_split=None,
)

# Initialize Custom Model (sklearn)
Reduction = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

In [42]:
# Train Models
print("=" * 50)
print("Training in process, please wait.")
Bottleneck.fit(X_train_scaled, y_train)
Parallel.fit(
    np.asarray(X_train_scaled, dtype="float32"),
    np.asarray(y_train, dtype="int64"),
)
Reduction.fit(X_train_scaled, y_train)
print("Training complete.")
print("=" * 50)

# Evaluate Models
b_y_pred = Bottleneck.predict(X_test_scaled)
p_y_pred = Parallel.predict(np.asarray(X_test_scaled, dtype="float32"))
r_y_pred = Reduction.predict(X_test_scaled)
print("=" * 50)
print("BOTTLNECK PERFORMANCE")
print("Test Accuracy:", accuracy_score(y_test, b_y_pred))
print("Classification Report:")
print(classification_report(y_test, b_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, b_y_pred))
print("=" * 50)
print("PARALLEL PERFORMANCE")
print("Test Accuracy:", accuracy_score(y_test, p_y_pred))
print("Classification Report:")
print(classification_report(y_test, p_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, p_y_pred))
print("=" * 50)
print("REDUCTION PERFORMANCE")
print("Test Accuracy:", accuracy_score(y_test, r_y_pred))
print("Classification Report:")
print(classification_report(y_test, r_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, r_y_pred))


Training in process, please wait.
Re-initializing module because the following parameters were re-set: input_dim, num_classes, pre_layers.
Re-initializing criterion.
Re-initializing optimizer.
  epoch    train_loss     dur
-------  ------------  ------
      1        2.3073  0.0979
      2        2.2882  0.0815
      3        2.1483  0.0670
      4        1.6931  0.0676
      5        1.2327  0.0889
      6        0.8775  0.0697
      7        0.6914  0.0646
      8        0.5327  0.0648
      9        0.4376  0.0652
     10        0.3431  0.0626
     11        0.2716  0.0630
     12        0.1962  0.0633
     13        0.1484  0.0678
     14        0.1120  0.0631
     15        0.0824  0.0633
     16        0.0623  0.0646
     17        0.0525  0.0718
     18        0.0419  0.0660
     19        0.0341  0.0632
     20        0.0349  0.0776
     21        0.0350  0.0658
     22        0.0292  0.0682
     23        0.0201  0.0649
     24        0.0154  0.0650
     25        0.0120  0.06